In [1]:
from datetime import datetime

In [2]:
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")

    return datetime.now().strftime(date_format)

In [3]:
get_current_datetime("%H:%M")

'00:32'

In [4]:
get_current_datetime_schema = {
    "name": "get_current_datetime",
    "description": "Get the current date and time in a specified format.",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "The datetime format to use, such as %Y-%m-%d %H:%M:%S or %H:%M."
            }
        },
        "required": []
    }
}

In [ ]:
from dotenv import load_dotenv
import os
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-5"

In [6]:
messages = [
    {
        "role": "user",
        "content": "What time is it?"
    }
]

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

response

Message(id='msg_011Cf2x7aWaPETNdkGrtwcCg', container=None, content=[ThinkingBlock(signature='EswCCpABCBEYAipAd7/zEl4SxSvLMcWXtlCbndTubvaTCufN9hFeKVOUWd67xt8ZQ1WrdXDDVAEvx3VDltlrBbtaBi5JEmbb6L4RnzIPY2xhdWRlLXNvbm5ldC01OABCCHRoaW5raW5nWiQ2YzJhNWE3Yy1mNzcwLTQ4NWUtOTE2MC1kZDBhNTZiMmJiYWWoAYbJntUGEgyS9MebAEYCvYcfwXQaDF7AZDsk8Wv4Ee1CWyIw9CJp2voyXj3Zzdkj/VUnwB46x2gvHPynLgvmQjqyH0UTAYOWxd+B7vj9/hajn/7cKmlKr4exyKmwkSg/hVJA4DEki/i45lbAmkINhCqDrInS+vPRaxITnuTaWh6fCXOFyjwWWlWBIDiw+KnGHVSD35RDJo0yMYuoXWropHmQXMoAkiCB4++0HsumZtZIlJ+gpBD3VyrbYTGddeQYAQ==', thinking='', type='thinking'), TextBlock(citations=None, text="I'll check the current time for you.", type='text'), ToolUseBlock(id='toolu_01YDNMW9raS46TwmHmAAd6ZB', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')], model='claude-sonnet-5', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephe

In [7]:
tool_use = next(
    block for block in response.content
    if block.type == "tool_use"
)

tool_result = get_current_datetime(
    tool_use.input.get("date_format", "%Y-%m-%d %H:%M:%S")
)

print(tool_result)

00:41:27


In [8]:
messages.append({
    "role": "assistant",
    "content": response.content
})

messages.append({
    "role": "user",
    "content": [
        {
            "type": "tool_result",
            "tool_use_id": tool_use.id,
            "content": tool_result
        }
    ]
})

final_response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

final_response

Message(id='msg_011Cf2xRsEuagvM47yKRvsvR', container=None, content=[TextBlock(citations=None, text='The current time is **00:41:27** (12:41:27 AM).', type='text')], model='claude-sonnet-5', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=587, output_tokens=23, output_tokens_details=OutputTokensDetails(thinking_tokens=0), server_tool_use=None, service_tier='standard'))

In [9]:
from datetime import timedelta

def add_duration_to_datetime(
    start_datetime,
    days=0,
    hours=0,
    minutes=0,
    seconds=0
):
    parsed_datetime = datetime.strptime(
        start_datetime,
        "%Y-%m-%d %H:%M:%S"
    )

    updated_datetime = parsed_datetime + timedelta(
        days=days,
        hours=hours,
        minutes=minutes,
        seconds=seconds
    )

    return updated_datetime.strftime("%Y-%m-%d %H:%M:%S")

In [10]:
add_duration_to_datetime(
    "2026-09-14 00:45:00",
    hours=2,
    minutes=30
)

'2026-09-14 03:15:00'

In [11]:
add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Add a duration to a starting date and time.",
    "input_schema": {
        "type": "object",
        "properties": {
            "start_datetime": {
                "type": "string",
                "description": "The starting date and time in YYYY-MM-DD HH:MM:SS format."
            },
            "days": {
                "type": "integer",
                "description": "Number of days to add."
            },
            "hours": {
                "type": "integer",
                "description": "Number of hours to add."
            },
            "minutes": {
                "type": "integer",
                "description": "Number of minutes to add."
            },
            "seconds": {
                "type": "integer",
                "description": "Number of seconds to add."
            }
        },
        "required": ["start_datetime"]
    }
}

In [12]:
messages = [
    {
        "role": "user",
        "content": "What date and time will it be 2 hours and 30 minutes after 2026-09-14 00:45:00?"
    }
]

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[
        get_current_datetime_schema,
        add_duration_to_datetime_schema
    ]
)

response

Message(id='msg_011Cf2y3Fx8FrHNusx3N5cGj', container=None, content=[ThinkingBlock(signature='EpQCCpABCBEYAipAtFcAHfikA1Ic9Kqe2Y3Qg4b7yqyiTOU3i61T/+rJ2dZTOoR85ubNbXqF51ZnPUpMF7Hb0GTEExwuVbD+03/0iTIPY2xhdWRlLXNvbm5ldC01OABCCHRoaW5raW5nWiQ2YzJhNWE3Yy1mNzcwLTQ4NWUtOTE2MC1kZDBhNTZiMmJiYWWoAd/OntUGEgxVk0v/IxxS2YdSuLgaDB7zKdOiv/4j6i6SXSIwYBVPPzn3IVEO91Jg903/kfku71ArISRyoQY/DQ9l30V/E6dMGKPpVjLgCznTp65SKjFSwDbXnCBRhCQeq6T02R0m2qFPIb4KPE+q66rsJNTVOcoE3gmZp+1Ph0tQSU0pPajFGAE=', thinking='', type='thinking'), ToolUseBlock(id='toolu_01U3uB7YyMJSQfDhiVxvGtUA', caller=DirectCaller(type='direct'), input={'start_datetime': '2026-09-14 00:45:00', 'hours': 2, 'minutes': 30}, name='add_duration_to_datetime', type='tool_use')], model='claude-sonnet-5', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inf

In [13]:
tool_use = next(
    block for block in response.content
    if block.type == "tool_use"
)

tool_result = add_duration_to_datetime(
    start_datetime=tool_use.input["start_datetime"],
    days=tool_use.input.get("days", 0),
    hours=tool_use.input.get("hours", 0),
    minutes=tool_use.input.get("minutes", 0),
    seconds=tool_use.input.get("seconds", 0)
)

print(tool_result)

2026-09-14 03:15:00


In [14]:
messages.append({
    "role": "assistant",
    "content": response.content
})

messages.append({
    "role": "user",
    "content": [
        {
            "type": "tool_result",
            "tool_use_id": tool_use.id,
            "content": tool_result
        }
    ]
})

final_response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[
        get_current_datetime_schema,
        add_duration_to_datetime_schema
    ]
)

final_response

Message(id='msg_011Cf2yQHt3kqvJVjv53UNLN', container=None, content=[TextBlock(citations=None, text='2 hours and 30 minutes after 2026-09-14 00:45:00 will be **2026-09-14 03:15:00**.', type='text')], model='claude-sonnet-5', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=897, output_tokens=45, output_tokens_details=OutputTokensDetails(thinking_tokens=0), server_tool_use=None, service_tier='standard'))

In [15]:
reminders = []

def set_reminder(reminder_text, reminder_datetime):
    if not reminder_text.strip():
        raise ValueError("reminder_text cannot be empty")

    if not reminder_datetime.strip():
        raise ValueError("reminder_datetime cannot be empty")

    datetime.strptime(
        reminder_datetime,
        "%Y-%m-%d %H:%M:%S"
    )

    reminder = {
        "text": reminder_text,
        "datetime": reminder_datetime
    }

    reminders.append(reminder)

    return f"Reminder set for {reminder_datetime}: {reminder_text}"

In [16]:
set_reminder(
    "Review Claude tool-use notes",
    "2026-09-14 09:00:00"
)

'Reminder set for 2026-09-14 09:00:00: Review Claude tool-use notes'

In [17]:
reminders

[{'text': 'Review Claude tool-use notes', 'datetime': '2026-09-14 09:00:00'}]

In [18]:
set_reminder_schema = {
    "name": "set_reminder",
    "description": "Store a reminder for a specific date and time.",
    "input_schema": {
        "type": "object",
        "properties": {
            "reminder_text": {
                "type": "string",
                "description": "The reminder message or task."
            },
            "reminder_datetime": {
                "type": "string",
                "description": "The date and time for the reminder in YYYY-MM-DD HH:MM:SS format."
            }
        },
        "required": [
            "reminder_text",
            "reminder_datetime"
        ]
    }
}

In [19]:
messages = [
    {
        "role": "user",
        "content": "Set a reminder to review Claude tool-use notes on 2026-09-14 at 09:00:00."
    }
]

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[
        get_current_datetime_schema,
        add_duration_to_datetime_schema,
        set_reminder_schema
    ]
)

response

Message(id='msg_011Cf2zThzKvxQRofcB4TH3V', container=None, content=[ToolUseBlock(id='toolu_01EDzDtvvud3dHmWC1R3ijwC', caller=DirectCaller(type='direct'), input={'reminder_text': 'Review Claude tool-use notes', 'reminder_datetime': '2026-09-14 09:00:00'}, name='set_reminder', type='tool_use')], model='claude-sonnet-5', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=901, output_tokens=101, output_tokens_details=OutputTokensDetails(thinking_tokens=0), server_tool_use=None, service_tier='standard'))

In [20]:
tool_use = next(
    block for block in response.content
    if block.type == "tool_use"
)

tool_result = set_reminder(
    reminder_text=tool_use.input["reminder_text"],
    reminder_datetime=tool_use.input["reminder_datetime"]
)

print(tool_result)

Reminder set for 2026-09-14 09:00:00: Review Claude tool-use notes


In [22]:
messages.append({
    "role": "assistant",
    "content": response.content
})

messages.append({
    "role": "user",
    "content": [
        {
            "type": "tool_result",
            "tool_use_id": tool_use.id,
            "content": tool_result
        }
    ]
})

final_response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[
        get_current_datetime_schema,
        add_duration_to_datetime_schema,
        set_reminder_schema
    ]
)

final_response

Message(id='msg_011Cf31ASbGJjVjihYHhFE1j', container=None, content=[TextBlock(citations=None, text='Done! I\'ve set a reminder to "Review Claude tool-use notes" for September 14, 2026 at 9:00 AM. Let me know if you\'d like to adjust the time or add any details to it.', type='text')], model='claude-sonnet-5', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=1175, output_tokens=66, output_tokens_details=OutputTokensDetails(thinking_tokens=0), server_tool_use=None, service_tier='standard'))

In [23]:
def run_tool_request(user_message):
    messages = [
        {
            "role": "user",
            "content": user_message
        }
    ]

    response = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
        tools=[
            get_current_datetime_schema,
            add_duration_to_datetime_schema,
            set_reminder_schema
        ]
    )

    tool_use = next(
        (
            block for block in response.content
            if block.type == "tool_use"
        ),
        None
    )

    if tool_use is None:
        return response.content[0].text

    if tool_use.name == "get_current_datetime":
        tool_result = get_current_datetime(
            tool_use.input.get(
                "date_format",
                "%Y-%m-%d %H:%M:%S"
            )
        )

    elif tool_use.name == "add_duration_to_datetime":
        tool_result = add_duration_to_datetime(
            start_datetime=tool_use.input["start_datetime"],
            days=tool_use.input.get("days", 0),
            hours=tool_use.input.get("hours", 0),
            minutes=tool_use.input.get("minutes", 0),
            seconds=tool_use.input.get("seconds", 0)
        )

    elif tool_use.name == "set_reminder":
        tool_result = set_reminder(
            reminder_text=tool_use.input["reminder_text"],
            reminder_datetime=tool_use.input["reminder_datetime"]
        )

    else:
        return f"Unknown tool requested: {tool_use.name}"

    messages.append({
        "role": "assistant",
        "content": response.content
    })

    messages.append({
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": tool_result
            }
        ]
    })

    final_response = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
        tools=[
            get_current_datetime_schema,
            add_duration_to_datetime_schema,
            set_reminder_schema
        ]
    )

    text_blocks = [
        block.text
        for block in final_response.content
        if block.type == "text"
    ]

    return "\n".join(text_blocks)

In [24]:
run_tool_request("What time is it?")

'The current time is **01:23:46**.'

In [25]:
run_tool_request(
    "What date and time will it be 1 day and 4 hours after 2026-09-14 10:00:00?"
)

'1 day and 4 hours after 2026-09-14 10:00:00 will be **2026-09-15 14:00:00**.'

In [26]:
run_tool_request(
    "Set a reminder to review Claude tool-use notes on 2026-09-14 at 09:30:00."
)

'Your reminder has been set for **September 14, 2026 at 9:30 AM** to "Review Claude tool-use notes." Let me know if you\'d like to adjust the time or add any additional reminders!'

In [27]:
def run_requested_tool(tool_use):
    if tool_use.name == "get_current_datetime":
        return get_current_datetime(
            tool_use.input.get(
                "date_format",
                "%Y-%m-%d %H:%M:%S"
            )
        )

    elif tool_use.name == "add_duration_to_datetime":
        return add_duration_to_datetime(
            start_datetime=tool_use.input["start_datetime"],
            days=tool_use.input.get("days", 0),
            hours=tool_use.input.get("hours", 0),
            minutes=tool_use.input.get("minutes", 0),
            seconds=tool_use.input.get("seconds", 0)
        )

    elif tool_use.name == "set_reminder":
        return set_reminder(
            reminder_text=tool_use.input["reminder_text"],
            reminder_datetime=tool_use.input["reminder_datetime"]
        )

    else:
        raise ValueError(
            f"Unknown tool requested: {tool_use.name}"
        )

In [28]:
def run_multi_turn_tool_request(user_message):
    messages = [
        {
            "role": "user",
            "content": user_message
        }
    ]

    while True:
        response = client.messages.create(
            model=model,
            max_tokens=1000,
            messages=messages,
            tools=[
                get_current_datetime_schema,
                add_duration_to_datetime_schema,
                set_reminder_schema
            ]
        )

        messages.append({
            "role": "assistant",
            "content": response.content
        })

        tool_uses = [
            block for block in response.content
            if block.type == "tool_use"
        ]

        if not tool_uses:
            text_blocks = [
                block.text
                for block in response.content
                if block.type == "text"
            ]

            return "\n".join(text_blocks)

        tool_result_blocks = []

        for tool_use in tool_uses:
            tool_result = run_requested_tool(tool_use)

            tool_result_blocks.append({
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": tool_result
            })

        messages.append({
            "role": "user",
            "content": tool_result_blocks
        })

In [29]:
run_multi_turn_tool_request(
    "What date will it be 103 days from today?"
)

'103 days from today (2026-09-14) will be **December 26, 2026**.'

In [30]:
def run_multi_turn_tool_request(user_message):
    messages = [
        {
            "role": "user",
            "content": user_message
        }
    ]

    while True:
        response = client.messages.create(
            model=model,
            max_tokens=1000,
            messages=messages,
            tools=[
                get_current_datetime_schema,
                add_duration_to_datetime_schema,
                set_reminder_schema
            ]
        )

        messages.append({
            "role": "assistant",
            "content": response.content
        })

        tool_uses = [
            block for block in response.content
            if block.type == "tool_use"
        ]

        if not tool_uses:
            text_blocks = [
                block.text
                for block in response.content
                if block.type == "text"
            ]

            return "\n".join(text_blocks)

        tool_result_blocks = []

        for tool_use in tool_uses:
            try:
                tool_result = run_requested_tool(tool_use)

                tool_result_blocks.append({
                    "type": "tool_result",
                    "tool_use_id": tool_use.id,
                    "content": tool_result,
                    "is_error": False
                })

            except Exception as error:
                tool_result_blocks.append({
                    "type": "tool_result",
                    "tool_use_id": tool_use.id,
                    "content": f"Error: {error}",
                    "is_error": True
                })

        messages.append({
            "role": "user",
            "content": tool_result_blocks
        })

In [31]:
run_multi_turn_tool_request(
    "What date will it be 45 days from today?"
)

'45 days from today (2026-09-14) will be **October 29, 2026**.'

In [32]:
run_multi_turn_tool_request(
    "What date and time will it be 2 hours after banana?"
)

'I\'d love to help, but "banana" isn\'t a valid date or time reference — I need a specific starting date/time (like "2025-06-15 14:30:00") to calculate 2 hours from.\n\nCould you clarify what starting date/time you meant? If you\'d like to use the current moment as the starting point, just let me know and I can fetch that and add 2 hours to it.'

In [33]:
from types import SimpleNamespace

bad_tool_use = SimpleNamespace(
    name="add_duration_to_datetime",
    input={
        "start_datetime": "banana",
        "hours": 2
    },
    id="test_tool_call"
)

try:
    tool_result = run_requested_tool(bad_tool_use)

    test_result_block = {
        "type": "tool_result",
        "tool_use_id": bad_tool_use.id,
        "content": tool_result,
        "is_error": False
    }

except Exception as error:
    test_result_block = {
        "type": "tool_result",
        "tool_use_id": bad_tool_use.id,
        "content": f"Error: {error}",
        "is_error": True
    }

test_result_block

{'type': 'tool_result',
 'tool_use_id': 'test_tool_call',
 'content': "Error: time data 'banana' does not match format '%Y-%m-%d %H:%M:%S'",
 'is_error': True}

In [34]:
run_multi_turn_tool_request(
    "Set a reminder for my doctor's appointment. It is 177 days after January 1, 2050."
)

'Your reminder has been set for **June 27, 2050 at 9:00 AM** — "Doctor\'s appointment." Let me know if you\'d like to adjust the time or add any additional details!'